# Notebook for Collisions

In [1]:
import sympy as sp
import numpy as np
from IPython.display import display_latex

In [2]:
## Helper functions
def sp_tensor(f, f_shape):
    """
    A helper function to help create sympy arrays based on the output of a function.

    f should be a function of the form f(*indices) where indices is a tuple of integers. f_shape
    describes the range over which each of the indices should range. For example, if f_shape is (3, 4),
    then the first index should range from 0 to 2 and the second index should range from 0 to 3.
    """

    index_arrays = np.meshgrid(*[np.arange(s) for s in f_shape], indexing='ij')
    view_arrays = [map(int, arr.flatten()) for arr in index_arrays]
    mapped_array = np.array([f(*indices) for indices in zip(*view_arrays)])

    return sp.Array(mapped_array.reshape(f_shape))

def sp_einsum(*args, **kwargs):
    """
    A helper function to help create sympy arrays based on the output of a function.

    This is a wrapper around np.einsum that allows for the output to be a sympy array instead of a numpy array.
    """

    return sp.Array(np.einsum(*args, **kwargs, optimize=True))

In [3]:
## Diffusion tensor in cartesian coordinates
# type: ignore

# momentum in cartesian coordinates
pc = sp.Array(sp.symbols('p_x p_y p_z', real=True))
p2 = sum(p**2 for p in pc)

# diffusion coefficients
dpar, dperp = sp.symbols(R'D_\parallel D_\perp', positive=True)


# gyrophase coordinates
ppar, pperp, pzeta = sp.symbols(R'p_\parallel p_\perp \zeta', real=True)
to_gy = lambda x: sp.simplify(x.subs({pc[2]: ppar, pc[0]: pperp * sp.cos(pzeta), pc[1]: pperp * sp.sin(pzeta)}))

# diffusion tensor in cartesian coordinates
dc = sp.simplify((dperp * (sp.Array(np.eye(3, dtype=int))*p2 - sp.tensorproduct(pc, pc)) + dpar * sp.tensorproduct(pc, pc)) / p2)

# diffusion vector fields
sigma0 = sp.sqrt(2*dperp / p2) * sp.Array([     0, -pc[2],  pc[1]])
sigma1 = sp.sqrt(2*dperp / p2) * sp.Array([ pc[2],      0, -pc[0]])
sigma2 = sp.sqrt(2*dperp / p2) * sp.Array([-pc[1],  pc[0],      0])
sigma3 = sp.sqrt(2*dpar  / p2) * sp.Array([ pc[0],  pc[1],  pc[2]])
sigma = [sigma0, sigma1, sigma2, sigma3]

dc_sigma = sum((sp.tensorproduct(s, s) for s in sigma), start=sp.Array(np.zeros((3, 3)))) / 2

display_latex('Show that the diffusion tensor can be expressed as the sum of the tensor products of the diffusion vector fields:', raw=True)
display_latex(to_gy(dc))
display_latex(sp.simplify(dc_sigma-dc))

In [4]:
## Diffusion tensor in magnetic coordinates
# type: ignore

# magnetic field and mass
B, m = sp.symbols('B m')

# magnetic coordinates
vpar = pc[2] / m
mu = (pc[0]**2 + pc[1]**2) / (2*m*B)
xi = sp.atan2(pc[1], pc[0])
vm = sp.Array([vpar, mu])

# Jacobian of the transformation from cartesian to magnetic coordinates
jac_mag = sp_tensor(lambda i, j: sp.diff(vm[i], pc[j]), (2,3))


# diffusion tensor in magnetic coordinates
dm = sp_einsum('ij,jk,lk->il', jac_mag, dc, jac_mag)

# Check against the diffusion tensor in Hirvijoki et al 2013
ke = p2 / (2*m)
dm_hirv = sp.Array(
    [[
        dpar / m**2 + (dperp - dpar) * mu * B / (m**2 * ke),
        mu * vpar / (m * ke) * (dpar - dperp),
    ], [
        mu * vpar / (m * ke) * (dpar - dperp),
        2 * mu / (m*B) * ((dpar - dperp)*mu*B/ke + dperp),
    ]])

display_latex('Show that the diffusion tensor in magnetic coordinates matches the expression in Hirvijoki et al 2013:', raw=True)
display_latex(to_gy(dm))
display_latex(sp.simplify(dm-dm_hirv))

for s in sigma:
    display_latex(to_gy(sp_einsum('ij,j->i', jac_mag, s)))

In [103]:
to_gy(sigma3)

[sqrt(2)*sqrt(D_\parallel)*p_\perp*cos(\zeta)/sqrt(p_\parallel**2 + p_\perp**2), sqrt(2)*sqrt(D_\parallel)*p_\perp*sin(\zeta)/sqrt(p_\parallel**2 + p_\perp**2), sqrt(2)*sqrt(D_\parallel)*p_\parallel/sqrt(p_\parallel**2 + p_\perp**2)]

In [28]:
## Add guiding center displacement to the diffusion vector fields
# type: ignore

Omega = sp.symbols('Omega')

sigma_gc = list(sp.Array([s[0], s[1], s[2], -s[1], s[0], 0]) for s in sigma)

xc = sp.Array(sp.symbols('x y z', real=True))
pcx = sp.Array([pc[0], pc[1], pc[2], xc[0], xc[1], xc[2]])
vmx = sp.Array([vpar, mu, xi, xc[0], xc[1], xc[2]])
jac_magx = sp_tensor(lambda i, j: sp.diff(vmx[i], pcx[j]), (6,6))


dx_sigma = sp.Array(np.zeros((6, 6), dtype=int))

for s in sigma_gc[:3]:
    display_latex(to_gy(s))
    display_latex(to_gy(sp.tensorproduct(s, s) / 2))
    dx_sigma += sp.tensorproduct(s, s) / 2

display_latex(to_gy(dx_sigma))

# hirvijoki spatial diffusion tensor
dx = to_gy((dpar - dperp) * (mu * B / (2*ke)) + dperp)

display_latex(sp.expand(dx))

In [98]:
sp.expand(dx)

D_\parallel*p_\perp**2/(2*p_\parallel**2 + 2*p_\perp**2) + D_\perp*p_\parallel**2/(p_\parallel**2 + p_\perp**2) + D_\perp*p_\perp**2/(2*p_\parallel**2 + 2*p_\perp**2)

In [95]:
s = sigma_gc[3] + sp.sqrt(dpar/dperp)*sigma_gc[2]
display_latex(to_gy(sp.tensorproduct(s,s)))

In [ ]:
# type: ignore

# Set up spherical coordinates
# magnitude of velocity
vmag = sp.sqrt(sum(v**2 for v in pc))
# cosine of the polar angle
xi = pc[0] / vmag
# azimuthal angle
phi = sp.atan2(pc[1], pc[2])

vsph = sp.Array([vmag, xi])

jacobian = sp.Array([[sp.diff(var, v) for v in pc] for var in vsph])

In [4]:
dsph = sp.Array([[sum(jacobian[i,k] * dc[k,l] * jacobian[j,l] for k in range(3) for l in range(3)) for i in range(2)] for j in range(2)])

In [5]:
sp.simplify(dsph)

[[\kappa_\parallel*(v_0**2 + v_1**2 + v_2**2), 0], [0, \kappa_\perp*(v_1**2 + v_2**2)/(v_0**2 + v_1**2 + v_2**2)]]

In [11]:
# check diffusion tensor in GC coordinates
p, xi = sp.symbols(R'p \xi')
m, B = sp.symbols('m B')
vpar = p * xi / m
mu = p**2 * (1 - xi**2) / (2 * m * B)
jacobian = sp.Array([[sp.diff(var, v) for v in [p, xi]] for var in [vpar, mu]])

dpar, dperp = sp.symbols(R'D_\parallel D_\perp')
# Lower block of the matrix in equation (49)
dsph = sp.Array([[dpar, 0], [0, (1-xi**2)*dperp/p**2]])

# Compute the transform from (p, xi) to (vpar, mu)
d1 = sp.Array([[sum(jacobian[i,k] * dsph[k,l] * jacobian[j,l] for k in range(2) for l in range(2)) for i in range(2)] for j in range(2)])

# Write the expression in eq (57)
K = p**2 / (2*m)
d2 = sp.Array([
    [dpar / m**2 + (dperp - dpar)/m**2 * mu * B / K,
     mu * vpar / (m * K) * (dpar - dperp)],
    [mu * vpar / (m * K) * (dpar - dperp),
     (K/B)**2 * 4 / p**2 * (1-xi**2) * (dpar*(1-xi**2) + dperp*xi**2)]
    ])

# Compare
display_latex(vpar)
display_latex(mu)

In [19]:
kpar, kperp = sp.symbols(R'\kappa_\parallel \kappa_\perp')
ksph = sp.Array([[kpar, 0], [0, kperp]])
kappa = sp.Array([[sum(jacobian[i,k] * ksph[k,j] for k in range(2)) for i in range(2)] for j in range(2)])
kappa2 = sp.Array([[kpar * xi / m, kperp * p / m], [kpar * mu / p, -kperp * xi * p**2 / m / B]])

display_latex(sp.transpose(kappa))
display_latex(kappa2)